# PS S6E5 - 051 RealMLP watchlist updated shared001v2 min

Experiment ID: `exp_20260525_051_realmlp_watchlist_updated_shared001v2_min`

Purpose:
- Reproduce the updated watchlist RealMLP / PyTabKit notebook as-is as much as possible.
- Keep the updated shared_001/shared_001v2-style feature engineering and RealMLP params unchanged.
- Add experiment-management outputs: OOF/PRED npy, submission, `cv_result.json`, `memo_draft.yml`, feature columns.

Notes:
- This is not a new feature-engineering experiment.
- `Normalized_TyreLife` is dropped from original data.
- Public reference reported for the updated source notebook: CV around 0.95409 / LB 0.95407.


In [1]:
%%time
!pip install -q pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 17.3 MB/s eta 0:00:00
CPU times: user 49.2 ms, sys: 21.6 ms, total: 70.8 ms
Wall time: 5.28 s


In [2]:
import os
import json
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from colorama import Fore, Style
from importlib.metadata import version
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

import torch
import pytabkit
from pytabkit import RealMLP_TD_Classifier

warnings.filterwarnings('ignore')
print("PyTorch  version:", torch.__version__)
print("PyTabKit version:", version("pytabkit"))

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)


PyTorch  version: 2.10.0+cu128
PyTabKit version: 1.7.3


In [3]:
# Experiment management config
EXP_ID = "exp_20260525_051_realmlp_watchlist_updated_shared001v2_min"
COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
METRIC = "AUC"
CREATED_AT = "2026-05-25"

OUTDIR = Path("/kaggle/working") / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

print("EXP_ID:", EXP_ID)
print("OUTDIR:", OUTDIR)


EXP_ID: exp_20260525_051_realmlp_watchlist_updated_shared001v2_min
OUTDIR: /kaggle/working/exp_20260525_051_realmlp_watchlist_updated_shared001v2_min


## Load Data

In [4]:
# Data paths
COMP_DATA_DIR = Path("/kaggle/input/competitions/playground-series-s6e5")
ORIG_DATA_PATH = Path("/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv")

train = pd.read_csv(COMP_DATA_DIR / "train.csv")
test = pd.read_csv(COMP_DATA_DIR / "test.csv")
orig = pd.read_csv(ORIG_DATA_PATH)

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Orig shape :", orig.shape)


Train shape: (439140, 16)
Test shape : (188165, 15)
Orig shape : (101371, 16)


## Preprocess Features

In [5]:
%%time
ID = 'id'
TARGET = 'PitNextLap'
orig = orig.drop(['Normalized_TyreLife'], axis=1, errors='ignore')
y_orig = orig[TARGET]; orig = orig.drop([TARGET], axis=1)
X = train.drop([ID, TARGET], axis=1); train_id = train[ID]
y = train[TARGET]
X_test = test.drop([ID], axis=1); test_id = test[ID]
del train, test
print("X      init shape:", X.shape)
print("X_test init shape:", X_test.shape)
print("orig   init shape:", orig.shape, "\n")

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols), "\n")

category_map = {}
important_combos = [
    ('Race', 'Compound'),
    ('Race', 'Year'),
]
def feature_engineering(df, fit=False):
    # Arithmetic interaction
    df['_LapNumber_/_RaceProgress'] = (df['LapNumber'] / (df['RaceProgress'] + 1e-6)).astype('float32')
    df['_TyreLife_/_LapNumber'] = (df['TyreLife'] / df['LapNumber'].clip(lower=1)).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation'] = (df['LapTime (s)'] * df['Cumulative_Degradation']).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation_abs'] = (df['LapTime (s)'] * df['Cumulative_Degradation'].abs()).astype('float32')
    df['_LapTime (s)_/_Cumulative_Degradation_abs'] = (df['LapTime (s)'] / (df['Cumulative_Degradation'].abs() + 1e-6)).astype('float32')

    # Categorize numericals
    for col in num_cols + ['_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber']:
        cat_name = f"{col}_cat_" if col in num_cols else f"{col[1:]}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype(str)

    # Count encoding
    for col in cat_cols + ['Year_cat_', 'PitStop_cat_']:
        count_name = f"_{col}_count" if col in cat_cols else f"_{col[:-1]}_count"
        if fit:
            count_map = df[col].value_counts()
            category_map[count_name] = count_map
        else:
            count_map = category_map[count_name]
        df[count_name] = df[col].map(count_map).fillna(0).astype('int32')

    # Discretize numericals
    bin_config = {'RaceProgress': [200], 'LapTime (s)': [7]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            for strategy in ['quantile']:
                bin_name = f"{col}_{n_bins}_{strategy}_bin_"
                if fit:
                    kb = KBinsDiscretizer(
                        n_bins=n_bins,
                        encode='ordinal',
                        strategy=strategy,
                        subsample=None
                    )
                    binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                    category_map[bin_name] = kb
                else:
                    kb = category_map[bin_name]
                    binned = kb.transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = binned
                df[bin_name] = df[bin_name].astype(str)

    # Create interaction categories
    combo_names = []
    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'
        combo_names.append(combo_name)
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + '_' + df[col].astype(str)
        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes
        df[combo_name] = df[combo_name].astype(str)   

    new_cat_cols = [col for col in df.columns if col.endswith('_')]
    new_num_cols = [col for col in df.columns if col.startswith('_')]
    return df, new_cat_cols, new_num_cols, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, new_cat_cols, new_num_cols, combo_names = feature_engineering(X_test, fit=False)
orig, new_cat_cols, new_num_cols, combo_names = feature_engineering(orig, fit=False)
cat_cols += new_cat_cols; num_cols += new_num_cols
print("len(new_cat_cols):", len(new_cat_cols))
print("len(new_num_cols):", len(new_num_cols), "\n")

print("prep len(cat_cols):", len(cat_cols))
print("prep len(num_cols):", len(num_cols), "\n")
print("X      prep shape:", X.shape)
print("X_test prep shape:", X_test.shape)
print("orig   prep shape:", orig.shape, "\n")
# Save base feature columns after source feature engineering, before fold-wise TE columns.
base_feature_columns = X.columns.tolist()
pd.DataFrame({"feature": base_feature_columns}).to_csv(OUTDIR / "feature_columns_base_before_te.csv", index=False)
print("Saved base feature columns:", OUTDIR / "feature_columns_base_before_te.csv")


X      init shape: (439140, 14)
X_test init shape: (188165, 14)
orig   init shape: (101371, 14) 

init len(cat_cols): 3
init len(num_cols): 11 

len(new_cat_cols): 17
len(new_num_cols): 10 

prep len(cat_cols): 20
prep len(num_cols): 21 

X      prep shape: (439140, 41)
X_test prep shape: (188165, 41)
orig   prep shape: (101371, 41) 

Saved base feature columns: /kaggle/working/exp_20260525_051_realmlp_watchlist_updated_shared001v2_min/feature_columns_base_before_te.csv
CPU times: user 3.56 s, sys: 367 ms, total: 3.93 s
Wall time: 3.95 s


## Config

In [6]:
class CFG:
    FOLDS = 5
    SEED = 42
    TE = True

# Keep updated watchlist notebook parameters as-is.
params = {
    'random_state': 42,
    'verbosity': 2,
    'val_metric_name': '1-auc_ovr',

    'n_ens': 20,
    'n_epochs': 5,
    'batch_size': 256,
    'use_early_stopping': False,
    'early_stopping_additive_patience': 10,
    'early_stopping_multiplicative_patience': 1,

    'lr': 0.019,
    'wd': 0.01,
    'sq_mom': 0.99,
    'lr_sched': 'lin_cos_log_15',
    'first_layer_lr_factor': 0.25,

    'embedding_size': 6,
    'max_one_hot_cat_size': 18,
    'hidden_sizes': [512, 256, 128],
    'act': 'silu',
    'p_drop': 0.05,
    'p_drop_sched': 'invsqrtp1e-3',

    'plr_hidden_1': 16,
    'plr_hidden_2': 8,
    'plr_act_name': 'gelu',
    'plr_lr_factor': 0.1151,
    'plr_sigma': 2.33,

    'ls_eps': 0.01,
    'ls_eps_sched': 'sqrt_cos',

    'add_front_scale': False,
    'bias_init_mode': 'neg-uniform-dynamic-2',
    'tfms': ['one_hot', 'median_center', 'robust_scale',
             'smooth_clip', 'embedding', 'l2_normalize'],
}

REFERENCE = {
    "reported_source_cv_auc": 0.95409,
    "reported_source_public_lb": 0.95407,
    "current_best_attack_add050_nm_cv_auc": 0.9550174162126694,
    "current_best_attack_add050_nm_public_lb": 0.95433,
    "current_best_stable_add050_logreg_cv_auc": 0.9549943053114434,
    "current_best_stable_add050_logreg_public_lb": 0.95430,
}


## Train K-Fold

In [7]:
%%time
skf = StratifiedKFold(n_splits=CFG.FOLDS, shuffle=True, random_state=CFG.SEED)
oof_preds = np.zeros(len(X), dtype=np.float32)
test_preds = np.zeros(len(X_test), dtype=np.float32)
fold_scores = []
fold_details = []
final_feature_columns = None

for fold, ((tr_idx, val_idx), (or_tr_idx, or_val_idx)) in enumerate(
          zip(skf.split(X, y), skf.split(orig, y_orig)), 1):
    X_tr = X.iloc[tr_idx].copy()
    orig_tr = orig.iloc[or_tr_idx].copy()
    X_tr = pd.concat([X_tr, orig_tr], axis=0).reset_index(drop=True)
    y_tr = pd.concat([y.iloc[tr_idx], y_orig.iloc[or_tr_idx]], axis=0).reset_index(drop=True)
    X_val = X.iloc[val_idx].copy()
    y_val = y.iloc[val_idx]
    X_tst = X_test.copy()

    # Target encoding. Kept as source notebook behavior.
    if CFG.TE:
        te_cols = combo_names
        TE = TargetEncoder(cv=CFG.FOLDS, smooth='auto', shuffle=True, random_state=CFG.SEED)
        tr_enc = TE.fit_transform(X_tr[te_cols], y_tr)
        val_enc = TE.transform(X_val[te_cols])
        tst_enc = TE.transform(X_tst[te_cols])

        te_names = [f"_{col}TE" for col in te_cols]
        X_tr[te_names] = tr_enc
        X_val[te_names] = val_enc
        X_tst[te_names] = tst_enc
    else:
        te_names = []

    if fold == 1:
        final_feature_columns = X_tr.columns.tolist()
        print("len(FEATURES):", len(final_feature_columns), "\n")
        pd.DataFrame({"feature": final_feature_columns}).to_csv(OUTDIR / "feature_columns.csv", index=False)

    print("#"*16)
    print(f"### Fold {fold}/{CFG.FOLDS} ...")
    print("#"*16)

    model = RealMLP_TD_Classifier(**params)
    model.fit(X_tr, y_tr, X_val, y_val)

    val_preds = model.predict_proba(X_val)[:, 1].astype(np.float32)
    fold_test_preds = model.predict_proba(X_tst)[:, 1].astype(np.float32)

    oof_preds[val_idx] = val_preds
    test_preds += fold_test_preds / CFG.FOLDS

    fold_score = roc_auc_score(y_val, val_preds)
    fold_scores.append(float(fold_score))
    fold_details.append({
        "fold": fold,
        "auc": float(fold_score),
        "n_train_comp": int(len(tr_idx)),
        "n_train_orig": int(len(or_tr_idx)),
        "n_valid": int(len(val_idx)),
    })
    print(f"{Fore.GREEN}{Style.BRIGHT}Fold {fold} | AUC Score: {fold_score:.5f}{Style.RESET_ALL}\n")
    torch.cuda.empty_cache()


len(FEATURES): 43 

################
### Fold 1/5 ...
################
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', '_LapTime (s)_*_Cumulative_Degradation', '_LapTime (s)_*_Cumulative_Degradation_abs', '_LapTime (s)_/_Cumulative_Degradation_abs', '_Driver_count', '_Compound_count', '_Race_count', '_Year_cat_count', '_PitStop_cat_count', '_Race_Compound_TE', '_Race_Year_TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'LapNumber_/_RaceProgress_cat_', 'TyreLife_/_LapNumber_cat_', 'RaceProgress_200_quantile_bin_', 'LapTime (s)_7_quantile_bin_', 'Race_Compound_

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/5: val 1-auc_ovr = 0.055358
Epoch 2/5: val 1-auc_ovr = 0.047382
Epoch 3/5: val 1-auc_ovr = 0.046350
Epoch 4/5: val 1-auc_ovr = 0.045306
Epoch 5/5: val 1-auc_ovr = 0.044830


`Trainer.fit` stopped: `max_epochs=5` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 1 | AUC Score: 0.95517

################
### Fold 2/5 ...
################
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', '_LapTime (s)_*_Cumulative_Degradation', '_LapTime (s)_*_Cumulative_Degradation_abs', '_LapTime (s)_/_Cumulative_Degradation_abs', '_Driver_count', '_Compound_count', '_Race_count', '_Year_cat_count', '_PitStop_cat_count', '_Race_Compound_TE', '_Race_Year_TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'LapNumber_/_RaceProgress_cat_', 'TyreLife_/_LapNumber_cat_', 'RaceProgress_200_quantile_bin_', 'LapTime (s)_7_quantile_bin_', 'Race_

...

Fold 4 | AUC Score: 0.95332

################
### Fold 5/5 ...
################
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', '_LapTime (s)_*_Cumulative_Degradation', '_LapTime (s)_*_Cumulative_Degradation_abs', '_LapTime (s)_/_Cumulative_Degradation_abs', '_Driver_count', '_Compound_count', '_Race_count', '_Year_cat_count', '_PitStop_cat_count', '_Race_Compound_TE', '_Race_Year_TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'LapNumber_/_RaceProgress_cat_', 'TyreLife_/_LapNumber_cat_', 'RaceProgress_200_quantile_bin_', 'LapTime (s)_7_quantile_bin_', 'Race_

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/5: val 1-auc_ovr = 0.055729
Epoch 2/5: val 1-auc_ovr = 0.047638
Epoch 3/5: val 1-auc_ovr = 0.046407
Epoch 4/5: val 1-auc_ovr = 0.045570
Epoch 5/5: val 1-auc_ovr = 0.045039


`Trainer.fit` stopped: `max_epochs=5` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 5 | AUC Score: 0.95496

CPU times: user 11min 7s, sys: 17.4 s, total: 11min 24s
Wall time: 11min 7s


## Evaluation and Submission

In [8]:
oof_score = float(roc_auc_score(y, oof_preds))
print("\n" + "="*24)
print(f"Overall OOF AUC: {Fore.BLACK}{Style.BRIGHT}{oof_score:.8f}{Style.RESET_ALL}")
print("="*24)

# Save OOF / pred in both npy and csv formats.
oof_path = OUTDIR / f"oof_{EXP_ID}.npy"
pred_path = OUTDIR / f"pred_{EXP_ID}.npy"
sub_path = OUTDIR / f"submission_{EXP_ID}.csv"
oof_csv_path = OUTDIR / f"oof_{EXP_ID}.csv"

np.save(oof_path, oof_preds.astype(np.float32))
np.save(pred_path, test_preds.astype(np.float32))

pd.DataFrame({ID: train_id, TARGET: oof_preds}).to_csv(oof_csv_path, index=False)
sub = pd.DataFrame({ID: test_id, TARGET: test_preds})
sub.to_csv(sub_path, index=False)

print("Saved:")
print(" OOF npy :", oof_path)
print(" PRED npy:", pred_path)
print(" OOF csv :", oof_csv_path)
print(" SUB     :", sub_path)
sub.head()



Overall OOF AUC: 0.95409327
Saved:
 OOF npy : /kaggle/working/exp_20260525_051_realmlp_watchlist_updated_shared001v2_min/oof_exp_20260525_051_realmlp_watchlist_updated_shared001v2_min.npy
 PRED npy: /kaggle/working/exp_20260525_051_realmlp_watchlist_updated_shared001v2_min/pred_exp_20260525_051_realmlp_watchlist_updated_shared001v2_min.npy
 OOF csv : /kaggle/working/exp_20260525_051_realmlp_watchlist_updated_shared001v2_min/oof_exp_20260525_051_realmlp_watchlist_updated_shared001v2_min.csv
 SUB     : /kaggle/working/exp_20260525_051_realmlp_watchlist_updated_shared001v2_min/submission_exp_20260525_051_realmlp_watchlist_updated_shared001v2_min.csv


,id,PitNextLap
0,439140,0.003118
1,439141,0.015658
2,439142,0.006664
3,439143,0.249377
4,439144,0.773671


In [9]:
# Save cv_result.json and memo_draft.yml

def to_builtin(obj):
    if isinstance(obj, dict):
        return {str(k): to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_builtin(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

cv_result = {
    "experiment": {
        "id": EXP_ID,
        "competition": COMPETITION,
        "metric": METRIC,
        "created_at": CREATED_AT,
        "type": "single_model_reproduction",
        "model_family": "RealMLP / PyTabKit",
    },
    "source": {
        "description": "Updated watchlist RealMLP / PyTabKit notebook, shared_001/shared_001v2 style.",
        "reported_cv_auc": REFERENCE["reported_source_cv_auc"],
        "reported_public_lb": REFERENCE["reported_source_public_lb"],
        "policy": "as-is reproduction with experiment-management outputs only",
    },
    "config": {
        "folds": CFG.FOLDS,
        "seed": CFG.SEED,
        "target_encoding": CFG.TE,
        "params": to_builtin(params),
    },
    "data": {
        "n_train": int(len(y)),
        "n_test": int(len(test_id)),
        "n_orig": int(len(y_orig)),
        "n_base_features_before_te": int(len(base_feature_columns)),
        "n_final_features_with_te": int(len(final_feature_columns)) if final_feature_columns is not None else None,
        "dropped_from_orig": ["Normalized_TyreLife"],
    },
    "result": {
        "cv_auc": oof_score,
        "fold_scores": fold_scores,
        "fold_details": fold_details,
        "public_lb": None,
    },
    "baselines": {
        "reported_source_cv_auc": REFERENCE["reported_source_cv_auc"],
        "reported_source_public_lb": REFERENCE["reported_source_public_lb"],
        "current_best_attack_add050_nm_cv_auc": REFERENCE["current_best_attack_add050_nm_cv_auc"],
        "current_best_attack_add050_nm_public_lb": REFERENCE["current_best_attack_add050_nm_public_lb"],
        "current_best_stable_add050_logreg_cv_auc": REFERENCE["current_best_stable_add050_logreg_cv_auc"],
        "current_best_stable_add050_logreg_public_lb": REFERENCE["current_best_stable_add050_logreg_public_lb"],
    },
    "artifacts": {
        "outdir": str(OUTDIR),
        "oof_npy": str(oof_path),
        "pred_npy": str(pred_path),
        "submission": str(sub_path),
        "oof_csv": str(oof_csv_path),
        "feature_columns": str(OUTDIR / "feature_columns.csv"),
        "feature_columns_base_before_te": str(OUTDIR / "feature_columns_base_before_te.csv"),
        "cv_result": str(OUTDIR / "cv_result.json"),
        "memo_draft": str(OUTDIR / "memo_draft.yml"),
    },
    "judgement": {
        "single_model_decision": "pending_public_lb",
        "next_step": "Submit single model if needed, then run Add051 blend using saved OOF/PRED.",
    },
}

cv_result_path = OUTDIR / "cv_result.json"
with open(cv_result_path, "w", encoding="utf-8") as f:
    json.dump(cv_result, f, indent=2, ensure_ascii=False)

memo_draft = f"""experiment:
  id: {EXP_ID}
  competition: {COMPETITION}
  metric: {METRIC}
  created_at: {CREATED_AT}
  type: single_model_reproduction
  status: completed

objective:
  summary: >
    Reproduce the updated watchlist RealMLP / PyTabKit shared_001/shared_001v2-style notebook
    as-is as much as possible, and add standard experiment-management outputs.
    The purpose is to test whether the reported CV around 0.95409 / Public LB around 0.95407
    can be reproduced and whether the model contributes to the current add050 stack.

source_reference:
  model_family: RealMLP / PyTabKit
  reported_cv_auc: {REFERENCE['reported_source_cv_auc']}
  reported_public_lb: {REFERENCE['reported_source_public_lb']}
  note: >
    Updated public/watchlist notebook reported stronger single-model performance than previous
    RealMLP-family single models, especially on Public LB.

implementation:
  policy: as_is_reproduction
  kept:
    - feature engineering from source notebook
    - RealMLP params from source notebook
    - FOLDS=5
    - SEED=42
    - original rows concatenated into fold train only
    - TargetEncoder for Race_Compound_ and Race_Year_
    - Normalized_TyreLife dropped from original data
  added_only:
    - EXP_ID / OUTDIR
    - OOF npy / pred npy / submission outputs
    - cv_result.json
    - memo_draft.yml
    - feature_columns.csv
  forbidden:
    - Normalized_TyreLife direct use
    - Normalized_TyreLife proxy construction
    - future endpoint proxy
    - pseudo label
    - extra feature engineering beyond source notebook

params:
  random_state: {params['random_state']}
  n_ens: {params['n_ens']}
  n_epochs: {params['n_epochs']}
  batch_size: {params['batch_size']}
  lr: {params['lr']}
  wd: {params['wd']}
  p_drop: {params['p_drop']}
  embedding_size: {params['embedding_size']}
  max_one_hot_cat_size: {params['max_one_hot_cat_size']}
  hidden_sizes: {params['hidden_sizes']}
  plr_lr_factor: {params['plr_lr_factor']}
  plr_sigma: {params['plr_sigma']}
  ls_eps: {params['ls_eps']}

result:
  cv_auc: {oof_score:.12f}
  public_lb: null
  fold_scores: {fold_scores}

comparison_targets:
  current_best_attack:
    name: add050_fixed_nm
    cv_auc: {REFERENCE['current_best_attack_add050_nm_cv_auc']}
    public_lb: {REFERENCE['current_best_attack_add050_nm_public_lb']}
  current_best_stable:
    name: add050_fixed_logreg
    cv_auc: {REFERENCE['current_best_stable_add050_logreg_cv_auc']}
    public_lb: {REFERENCE['current_best_stable_add050_logreg_public_lb']}

artifacts:
  outdir: {OUTDIR}
  oof_npy: {oof_path}
  pred_npy: {pred_path}
  submission: {sub_path}
  oof_csv: {oof_csv_path}
  cv_result: {cv_result_path}
  feature_columns: {OUTDIR / 'feature_columns.csv'}

success_criteria:
  single:
    minimum_cv_auc: 0.95404
    strong_public_lb: 0.95407
  blend:
    nm_cv_must_exceed: {REFERENCE['current_best_attack_add050_nm_cv_auc']}
    nm_public_lb_must_exceed: {REFERENCE['current_best_attack_add050_nm_public_lb']}
    logreg_public_lb_must_tie_or_exceed: {REFERENCE['current_best_stable_add050_logreg_public_lb']}

next_action:
  - Submit single model only if needed to verify Public LB.
  - Run Add051 blend with oof/pred from this experiment.
  - Compare weight, correlation, NM CV/Public LB, and LogReg CV/Public LB against add050 fixed.

judgement:
  decision: pending_public_lb_and_blend
  note: >
    Even if single-model Public LB is strong, this is likely a RealMLP-family near variant.
    Do not replace final candidates based on single score alone; use Add051 blend result.
"""

memo_path = OUTDIR / "memo_draft.yml"
with open(memo_path, "w", encoding="utf-8") as f:
    f.write(memo_draft)

print("Saved cv_result:", cv_result_path)
print("Saved memo_draft:", memo_path)


Saved cv_result: /kaggle/working/exp_20260525_051_realmlp_watchlist_updated_shared001v2_min/cv_result.json
Saved memo_draft: /kaggle/working/exp_20260525_051_realmlp_watchlist_updated_shared001v2_min/memo_draft.yml
